# T3 — Interactive Plotly Report · `HARD`

**Task:** Using **Plotly Express**, recreate 3 of your Matplotlib charts as interactive equivalents.  
Add custom hover data (at least **2 columns shown on hover**), color-encode a categorical variable,  
and **export each as a standalone HTML file**.  
Additionally, build a **4th interactive chart type not covered in class** (e.g., sunburst, treemap)  
with documented code explaining your design choices.


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px

# Using plotly_dark template to stay consistent with the overall dark theme
TEMPLATE = "plotly_dark"

# Load and prep Titanic
df = sns.load_dataset("titanic")
df["survived_label"] = df["survived"].map({0: "Did not survive", 1: "Survived"})
df["pclass_label"]   = df["pclass"].map({1: "1st Class", 2: "2nd Class", 3: "3rd Class"})
df["age"]            = df["age"].fillna(df["age"].median())  # fill NaN for hover display

print("Data ready:", df.shape)


In [ ]:
# ── Chart 1 · SCATTER — Age vs Fare (Interactive; replaces T1 Chart 4) ────────
#
# Design choices:
#   • bubble size = fare magnitude → encodes 3rd variable without colour collision
#   • symbol = sex → distinguishes gender within each survival group
#   • hover includes age, fare, class, sex, outcome — 5 columns of context
#
fig1 = px.scatter(
    df.dropna(subset=["age","fare"]),
    x="age", y="fare",
    color="survived_label",
    color_discrete_map={"Survived": "#2a9d8f", "Did not survive": "#e76f51"},
    symbol="sex",
    size="fare", size_max=18,          # bubble area proportional to fare
    hover_data={
        "age": True,
        "fare": ":.2f",
        "pclass_label": True,          # 2+ hover columns ✓
        "sex": True,
        "survived_label": True,
    },
    hover_name="pclass_label",
    labels={"age": "Age (years)", "fare": "Fare (£ GBP)",
            "survived_label": "Outcome", "sex": "Sex"},
    title="T3 · Chart 1 — Age vs Fare  |  hover a point for details",
    template=TEMPLATE, opacity=0.75,
)
fig1.write_html("t3_chart1_scatter.html")
fig1.show()
print("✅ Saved: t3_chart1_scatter.html")


In [ ]:
# ── Chart 2 · BAR — Survival count by class (Interactive; replaces T1 Chart 1) ─
#
# Grouped bar + hover shows absolute count, survival rate, and total per class
# so the viewer gets both raw numbers and percentages in one hover card.
#
surv = (df.groupby(["pclass_label","survived_label"])
          .agg(count=("survived","count"))
          .reset_index())
totals = df.groupby("pclass_label")["survived"].count().rename("total")
surv   = surv.merge(totals, on="pclass_label")
surv["rate"] = (surv["count"] / surv["total"] * 100).round(1)

fig2 = px.bar(
    surv,
    x="pclass_label", y="count",
    color="survived_label",
    color_discrete_map={"Survived": "#2a9d8f", "Did not survive": "#e76f51"},
    barmode="group",
    hover_data={"count": True, "rate": True, "total": True},  # 2+ hover columns ✓
    labels={"pclass_label": "Class", "count": "Passengers",
            "survived_label": "Outcome", "rate": "Survival Rate (%)"},
    title="T3 · Chart 2 — Passenger Count & Survival by Class",
    template=TEMPLATE,
)
fig2.write_html("t3_chart2_bar.html")
fig2.show()
print("✅ Saved: t3_chart2_bar.html")


In [ ]:
# ── Chart 3 · BOX — Fare by class (Interactive; replaces T1 Chart 5) ──────────
#
# Plotly's box shows individual outlier points on hover with full row metadata.
# points="outliers" keeps the chart clean — only extreme values are plotted.
#
fig3 = px.box(
    df,
    x="pclass_label", y="fare",
    color="pclass_label",
    color_discrete_sequence=["#2a9d8f", "#f4a261", "#e76f51"],
    points="outliers",
    hover_data=["sex", "age", "survived_label"],   # 2+ hover columns ✓
    labels={"pclass_label": "Passenger Class", "fare": "Fare (£ GBP)"},
    title="T3 · Chart 3 — Fare Distribution by Class",
    template=TEMPLATE,
)
fig3.write_html("t3_chart3_box.html")
fig3.show()
print("✅ Saved: t3_chart3_box.html")


In [ ]:
# ── Chart 4 · SUNBURST — not covered in class ─────────────────────────────────
#
# DESIGN CHOICES DOCUMENTED:
#
#   WHY SUNBURST (not treemap):
#     We have a strict 3-level hierarchy: Class → Sex → Outcome.
#     Sunburst's radial layout makes the drill-down path visually obvious.
#     Treemap uses area; sunburst uses arc angle — both encode quantity, but
#     treemap clusters rectangles in ways that break the hierarchy visually.
#
#   WHY COUNT (not rate) as values:
#     Rate would make all parent slices equal size, hiding the fact that
#     3rd class had the most passengers. Count keeps the chart proportional.
#
#   COLOR encodes Outcome (the most analytically important variable) at the
#     outermost ring so it's the first thing the eye lands on.
#
#   HOW TO READ:
#     Inner ring  = Passenger Class
#     Middle ring = Sex within each class
#     Outer ring  = Survived vs Did not survive
#     Click any segment to zoom in; click the centre to zoom out.
#
df_sun = (df.groupby(["pclass_label","sex","survived_label"])
            .agg(count=("survived","count"))
            .reset_index())
group_total        = df_sun.groupby(["pclass_label","sex"])["count"].transform("sum")
df_sun["rate"]     = (df_sun["count"] / group_total * 100).round(1)

fig4 = px.sunburst(
    df_sun,
    path=["pclass_label","sex","survived_label"],
    values="count",
    color="survived_label",
    color_discrete_map={
        "Survived":        "#2a9d8f",
        "Did not survive": "#e76f51",
        "(?)":             "#264653",   # Plotly assigns this to parent nodes
    },
    hover_data={"count": True, "rate": True},
    title=("T3 · Chart 4 — Sunburst: Class → Sex → Survival<br>"
           "<sup>Click to zoom in | click centre to zoom out</sup>"),
    template=TEMPLATE,
    maxdepth=3,
)
fig4.update_traces(textinfo="label+percent parent")
fig4.write_html("t3_chart4_sunburst.html")
fig4.show()
print("✅ Saved: t3_chart4_sunburst.html")
